# Moteur de recherche — Partie 3

Interface du moteur de recherche. Conformément aux consignes, **seule l'interface vit dans ce notebook** :
le chargement du corpus, l'indexation et la recherche sont réalisés par les modules `.py` de `source/backend`.

*Author: Corentin Legrand*

## 1. Préparation : réutiliser le moteur

### 1.1 Import des classes et chargement du corpus

Les modules du backend sont rendus importables en ajoutant `source/backend` au chemin de recherche de Python.
Le notebook doit être exécuté depuis son dossier (`source/frontend`), comportement par défaut de VS Code et Jupyter.

In [ ]:
import sys
from pathlib import Path

BACKEND_PATH: Path = Path.cwd().parent / "backend"
if str(BACKEND_PATH) not in sys.path:
    sys.path.insert(0, str(BACKEND_PATH))

In [ ]:
from helpers.environment_manager import EnvironmentManager
from infrastructure.json_dao import JsonDAO
from infrastructure.json_reader import JsonReader
from infrastructure.json_writer import JsonWriter
from models.corpus import Corpus
from models.document import Document
from script.scrap_data import DATA_FILENAME_KEY
from service.author_manager import AuthorManager
from service.corpus_manager import CorpusManager
from service.corpus_mapper import CorpusMapper
from service.document_factory import DocumentFactory
from service.search_engine import SearchEngine

Le corpus est rechargé depuis le fichier produit en Partie 1 (`load`) : les API Reddit et Arxiv ne sont pas
réinterrogées.

In [ ]:
data_filename: str = EnvironmentManager.get_instance().get_environment_variable(DATA_FILENAME_KEY)
mapper: CorpusMapper = CorpusMapper(DocumentFactory(), AuthorManager())
corpus_manager: CorpusManager = CorpusManager(JsonDAO(JsonWriter(), JsonReader()), mapper, data_filename)
corpus_manager.load()

corpus: Corpus = Corpus.get_instance()
first_document: Document = next(iter(corpus.documents.values()))
print(repr(corpus_manager))
print(f"First document: {first_document}")

### 1.2 et 1.3 Instanciation du moteur

La construction des matrices TF et TF×IDF affiche une barre de progression `tqdm` sur la boucle des documents
(`TermIndexBuilder`). La recherche elle-même est un produit matriciel, sans boucle Python à suivre.

In [ ]:
search_engine: SearchEngine = SearchEngine(corpus)

Vérification rapide par quelques requêtes en mots-clés (rappel de la Partie 2).

In [ ]:
from IPython.display import display

NB_RESULTS: int = 5
TEST_QUERIES: list[str] = ["climate change", "quantum error correction", "solar energy"]

for query in TEST_QUERIES:
    print(f"Query: {query}")
    display(search_engine.search(query, NB_RESULTS))

## 2. Interface graphique avec ipywidgets

### 2.1 Widgets simples
Un titre, un champ pour les mots-clés et un curseur pour le nombre de documents à retourner.

In [ ]:
import ipywidgets as widgets
import pandas as pd

from service.search_engine import EmptyQueryError

MIN_NB_DOCUMENTS: int = 1
MAX_NB_DOCUMENTS: int = 50
DEFAULT_NB_DOCUMENTS: int = 10
NO_RESULT_MESSAGE: str = "Aucun document ne correspond à ces mots-clés."

title: widgets.Label = widgets.Label(value="Moteur de recherche")
keywords_input: widgets.Text = widgets.Text(description="Mots clés", placeholder="ex. climate change")
nb_documents_slider: widgets.IntSlider = widgets.IntSlider(
    value=DEFAULT_NB_DOCUMENTS,
    min=MIN_NB_DOCUMENTS,
    max=MAX_NB_DOCUMENTS,
    description="Nombre d'articles à extraire",
    style={"description_width": "initial"},
)

### 2.3 et 2.4 Zone de résultats et bouton
La zone `Output` reçoit l'affichage de la recherche, déclenchée par le bouton via `on_click`.

In [ ]:
results_output: widgets.Output = widgets.Output()
search_button: widgets.Button = widgets.Button(description="Rechercher")

### 2.5 Fonction appelée au clic
Elle lit les mots-clés et le nombre de documents dans les widgets, appelle `search` du moteur et affiche le résultat
dans la zone `Output`, vidée à chaque appel.

In [ ]:
def on_search_button_click(button: widgets.Button) -> None:
    """Search the keywords typed by the user and display the results.

    Args:
        button (widgets.Button): The clicked button.
    """
    with results_output:
        results_output.clear_output()
        try:
            results: pd.DataFrame = search_engine.search(keywords_input.value, nb_documents_slider.value)
        except EmptyQueryError as error:
            print(error)
        else:
            if results.empty:
                print(NO_RESULT_MESSAGE)
            else:
                display(results)


search_button.on_click(on_search_button_click)

### 2.2 et 2.6 Affichage de l'interface
Les widgets sont empilés verticalement avec `VBox`, le champ et le bouton côte à côte avec `HBox`.

In [ ]:
display(widgets.VBox([title, widgets.HBox([keywords_input, search_button]), nb_documents_slider, results_output]))